### Utility function for taking input of samples

In [1]:
def input_data(n=None, dtype=float):
    data = []
    i = 0
    while True:
        if i==n: return data
        inp = input(f'Enter element {i+1}: ')
        if not inp and n is None: return data
        data.append(dtype(inp))
        i += 1

## 5a: Write a program to implement testing of hypothesis for one sample using Sign test.

In [2]:
from scipy.stats import binom, norm
from numbers import Number

def signtest(data1, data2, correction=True, alternative='two-sided'):
    data2 = [data2]*len(data1) if isinstance(data2, Number) else data2
    Np = Nn = 0
    for obs1, obs2 in zip(data1, data2, strict=True):
        if obs1-obs2<0: Nn += 1
        elif obs1-obs2>0: Np += 1
    
    m = Np+Nn
    cc = 0.5 if m>25 and correction else 0
    dist = norm(m/2, m**(1/2)/2) if m>25 else binom(m, 0.5)
    if alternative=='two-sided':
        return (S:=min(Nn, Np)), 2*dist.cdf(S+cc), m
    elif alternative=='greater':
        return Nn, dist.cdf(Nn+cc), m
    elif alternative=='less':
        return Np, dist.cdf(Np+cc), m
    else:
        raise ValueError("alternative must be 'less', 'greater' or 'two-sided'")

In [3]:
data1 = input_data()
if input('Population median or second set of observations?: ')=='med':
    data2 = float(input('Enter the population median: '))
else:
    data2 = input_data(len(data1))
correct = bool(input('Should the continuity correction be applied?: '))
alt = alt if (alt:=input('Enter the type of test: ')) else 'two-sided'

results = signtest(data1, data2, correction=correct, alternative=alt)

In [4]:
print('Data 1:', data1)
print('Data 2:', data2)
print(f'continuity correction: {correct}, alternative: {alt}')
results

Data 1: [5.0, 4.0, 6.0, 7.0, 3.0, 6.0, 4.0, 8.0, 9.0, 1.0, 2.0]
Data 2: 5.0
continuity correction: True, alternative: two-sided


(5, 1.24609375, 10)

## 5b: Write a program to implement testing of hypothesis for one sample using Wilcoxon Signed Rank test.

In [5]:
from scipy.stats import norm

def signedranktest(data1, data2, correction=False, alternative='two-sided'):
    d = [(di/abs(di), abs(di)) for elem1, elem2 
            in zip(data1, data2, strict=True) if (di:=elem1-elem2)!=0]
    sorted_d = sorted(d, key=lambda tup: tup[1]) + [(0, float('inf'))]
    
    Wp = Wn = 0
    prev = sorted_d[0][1]
    accum_ranks = []
    accum_signs = []
    for i, (sign, elem) in enumerate(sorted_d, start=1):
        if elem==prev:
            accum_ranks.append(i)
            accum_signs.append(sign)
        else:
            rank = sum(accum_ranks)/len(accum_ranks)
            for k in accum_signs:
                if k==1: Wp += rank
                else: Wn += rank
            accum_ranks = [i]
            accum_signs = [sign]
        prev = elem
    
    m = len(d)
    cc = 0.5 if correction else 0
    dist = norm(m*(m+1)/4, (m*(m+1)*(2*m+1)/24)**(1/2))
    if alternative=='two-sided':
        return (W:=min(Wn, Wp)), 2*dist.cdf(W+cc), m
    elif alternative=='greater':
        return Wn, dist.cdf(Wn+cc), m
    elif alternative=='less':
        return Wp, dist.cdf(Wp+cc), m
    else:
        raise ValueError("alternative must be 'less', 'greater' or 'two-sided'")

In [6]:
data1 = input_data()
data2 = input_data(len(data1))
correct = bool(input('Should the continuity correction be applied?: '))
alt = alt if (alt:=input('Enter the type of test: ')) else 'two-sided'

results = signedranktest(data1, data2, correction=correct, alternative=alt)

In [7]:
print('Sample 1:', data1)
print('Sample 2:', data2)
print(f'continuity correction: {correct}, alternative: {alt}')
results

Sample 1: [3.0, 5.0, 4.0, 3.0, 6.0, 7.0]
Sample 2: [3.0, 5.0, 4.0, 6.0, 7.0, 3.0]
continuity correction: False, alternative: two-sided


(3.0, 1.0, 3)

### Using scipy.stats module

In [8]:
from scipy.stats import wilcoxon
data1 = input_data()
data2 = input_data(len(data1))
correct = bool(input('Should the continuity correction be applied?: '))
alt = alt if (alt:=input('Enter the type of test: ')) else 'two-sided'

results = wilcoxon(data1, data2, correction=correct, alternative=alt, method='approx')

/Users/asadullahshaikh/.pyenv/versions/3.12.2/envs/venv/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:531: UserWarning: Sample size too small for normal approximation.
  res = hypotest_fun_out(*samples, **kwds)


In [9]:
print('Sample 1:', data1)
print('Sample 2:', data2)
print(f'continuity correction: {correct}, alternative: {alt}')

results

Sample 1: [3.0, 5.0, 4.0, 3.0, 6.0, 7.0]
Sample 2: [3.0, 5.0, 4.0, 6.0, 7.0, 3.0]
continuity correction: False, alternative: two-sided


WilcoxonResult(statistic=3.0, pvalue=1.0)

## 5c: Write a program to implement testing of hypothesis for one sample using Wilcoxon Rank Sum test/Mann-Whitney U test.

In [10]:
from scipy.stats import norm

def utest(data1, data2, correction=True, alternative='two-sided'):
    data1 = [(1, elem) for elem in data1]
    data2 = [(2, elem) for elem in data2]
    sorted_samples = sorted(data1+data2, key=lambda tup: tup[1]) + [(0, float('inf'))]
    
    R1 = R2 = 0
    prev = sorted_samples[0][1]
    accum_ranks = []
    accum_nums = []
    for i, (samp_num, elem) in enumerate(sorted_samples, start=1):
        if elem==prev:
            accum_ranks.append(i)
            accum_nums.append(samp_num)
        else:
            rank = sum(accum_ranks)/len(accum_ranks)
            for k in accum_nums:
                if k==1: R1 += rank
                else: R2 += rank
            accum_ranks = [i]
            accum_nums = [samp_num]
        prev = elem
    
    n1, n2 = len(data1), len(data2)
    U1, U2 = R1-(n1*(n1+1)/2), R2-(n2*(n2+1)/2)
    cc = 0.5 if correction else 0
    dist = norm(n1*n2/2, (n1*n2*(n1+n2+1)/12)**(1/2))
    if alternative=='two-sided':
        return (U:=min(U1, U2)), 2*dist.cdf(U+cc)
    elif alternative=='greater':
        return U2, dist.cdf(U2+cc)
    elif alternative=='less':
        return U1, dist.cdf(U1+cc)
    else:
        raise ValueError("alternative must be 'less', 'greater' or 'two-sided'")

In [11]:
data1 = input_data()
data2 = input_data()
correct = bool(input('Should the continuity correction be applied?: '))
alt = alt if (alt:=input('Enter the type of test: ')) else 'two-sided'

results = utest(data1, data2, correction=correct, alternative=alt)

In [12]:
print('Sample 1:', data1)
print('Sample 2:', data2)
print(f'continuity correction: {correct}, alternative: {alt}')
results

Sample 1: [3.0, 2.0, 5.0, 4.0, 6.0]
Sample 2: [2.0, 4.0, 3.0, 5.0, 6.0, 8.0, 6.0, 9.0]
continuity correction: False, alternative: two-sided


(13.0, 0.30550708686125383)

### Using scipy.stats module

In [13]:
from scipy.stats import mannwhitneyu
data1 = input_data()
data2 = input_data()
correct = bool(input('Should the continuity correction be applied?: '))
alt = alt if (alt:=input('Enter the type of test: ')) else 'two-sided'

results = mannwhitneyu(data1, data2, use_continuity=correct, alternative=alt, method='asymptotic')

In [14]:
print('Sample 1:', data1)
print('Sample 2:', data2)
print(f'continuity correction: {correct}, alternative: {alt}')
results

Sample 1: [3.0, 2.0, 5.0, 4.0, 6.0]
Sample 2: [2.0, 4.0, 3.0, 5.0, 6.0, 8.0, 6.0, 9.0]
continuity correction: False, alternative: two-sided


MannwhitneyuResult(statistic=13.0, pvalue=0.30013471623548893)